# ACT Silver Layer Pipeline

**Purpose:** Consolidate 2 bronze ACT tables into a single, clean silver table

**Source Tables:**
* `workspace.bronze.act` (2022-25, 7,170 rows)
* `workspace.bronze.act_highest` (2010-22, 31,361 rows)

**Output:** `workspace.silver.act` (38,531 rows, 20 columns)

---

## Key Transformations

### 1. Table Consolidation
* **Two source tables** merged with `unionByName`:
  * `act` = Recent data (2022-25)
  * `act_highest` = Historical data (2010-22)
* No overlap between tables - clean union

### 2. District Code Consolidation
* **Problem:** Inconsistent column names across source tables
  * Some files use `SCHOOL_DSTRCT_CD`
  * Others use `SCHOOL_DISTRCT_CD` (typo in source)
* **Solution:** `coalesce()` to merge both columns
* Ensures no data loss from column naming inconsistency

### 3. Institution Key
* **`institution_key`** = composite join key for linking to other tables
* **Format:** `district_code_institution_number`
* Used to join with demographics, attendance, graduation tables

### 4. Decimal Formatting Cleanup
* **Problem:** Count columns have `.0` suffix (stored as string "15.0")
* **Solution:** Strip `.0` with regexp_replace before casting to IntegerType
* Ensures clean integer values

### 5. TFS Suppression Handling
* **TFS** = "Too Few Students" (privacy threshold < 10 students)
* Converted to NULL for all numeric columns
* Preserves data integrity for aggregations

### 6. Type Casting
* **Counts:** Cast from string to IntegerType (after cleaning .0)
* **Scores:** Cast from string to DoubleType
* Ensures proper arithmetic operations in gold layer

---

## Schema

| Column | Type | Description |
|--------|------|-------------|
| school_year | string | Academic year (e.g., '2024-25') |
| institution_number | string | Institution number |
| institution_name | string | School name |
| district_code | string | District code (consolidated from 2 source columns) |
| district_name | string | District name |
| subgroup | string | Student subgroup (All Students, race/ethnicity, etc.) |
| test_component | string | ACT test component (Composite, English, Math, Reading, Science) |
| **institution_key** | string | Composite join key |
| national_num_tested | int | Count suppressed as NULL if TFS |
| state_num_tested | int | Count suppressed as NULL if TFS |
| district_num_tested | int | Count suppressed as NULL if TFS |
| institution_num_tested | int | Count suppressed as NULL if TFS |
| national_avg_score | double | Score suppressed as NULL if TFS |
| state_avg_score | double | Score suppressed as NULL if TFS |
| district_avg_score | double | Score suppressed as NULL if TFS |
| institution_avg_score | double | Score suppressed as NULL if TFS |
| assessment_code | string | Assessment code |
| highest_recent_indicator | string | 'Highest' or 'Recent' (from act_highest table) |
| source_year | string | Source file year |
| source_file | string | Source file name |

---

## ACT Test Components

**5 Components:**

1. **Composite** - Overall score (1-36 scale, average of 4 subject scores)
2. **English** - Grammar, usage, punctuation (1-36)
3. **Mathematics** - Algebra, geometry, trigonometry (1-36)
4. **Reading** - Comprehension, inference (1-36)
5. **Science** - Data interpretation, reasoning (1-36)

**Score Scale:** 1-36 (national average typically ~20-21)

In [0]:
# ACT Silver Layer Pipeline
# Purpose: Clean, type, and consolidate ACT test score data from bronze layer
# Output: workspace.silver.act

from pyspark.sql.functions import col, coalesce, when, regexp_replace, concat_ws, count as cnt
from pyspark.sql.types import DoubleType, IntegerType

print("=== ACT Silver Layer Pipeline ===\n")
print("Loading bronze tables...")

# Load source tables
act_recent = spark.table("workspace.bronze.act")
act_historical = spark.table("workspace.bronze.act_highest")

print(f"  Recent (2022-25): {act_recent.count():,} rows")
print(f"  Historical (2010-22): {act_historical.count():,} rows")

print("\n=== Step 1: Check for duplicates ===\n")

# Union tables first
act_all = act_recent.unionByName(act_historical, allowMissingColumns=True)
print(f"Total bronze rows after union: {act_all.count():,}\n")

# Business key: school_year + district + institution + subgroup + test_component
# Note: Using coalesce for district code due to column name inconsistency
duplicates = act_all.groupBy(
    'LONG_SCHOOL_YEAR', 
    coalesce(col('SCHOOL_DSTRCT_CD'), col('SCHOOL_DISTRCT_CD')).alias('district_code'),
    'INSTN_NUMBER', 
    'SUBGRP_DESC', 
    'TEST_CMPNT_TYP_CD'
).agg(cnt('*').alias('row_count')).filter('row_count > 1')

dupe_count = duplicates.count()
if dupe_count > 0:
    print(f"⚠️  Found {dupe_count:,} duplicate business keys")
    print("Sample duplicates:")
    duplicates.orderBy(col('row_count').desc()).show(5, truncate=False)
else:
    print("✓ No duplicates detected")

print("\n=== Step 2: Remove duplicates from source data ===\n")

before_dedup = act_all.count()
act_all = act_all.dropDuplicates([
    'LONG_SCHOOL_YEAR', 'SCHOOL_DSTRCT_CD', 'SCHOOL_DISTRCT_CD', 'INSTN_NUMBER', 
    'SUBGRP_DESC', 'TEST_CMPNT_TYP_CD'
])
after_dedup = act_all.count()

print(f"Rows before deduplication: {before_dedup:,}")
print(f"Rows after deduplication: {after_dedup:,}")
print(f"Removed {before_dedup - after_dedup:,} duplicate rows")

print("\n=== Step 3: Apply final transformations ===\n")

# Build silver table with cleaned/typed columns
act_silver = act_all.select(
    col('LONG_SCHOOL_YEAR').alias('school_year'),
    col('INSTN_NUMBER').alias('institution_number'),
    col('INSTN_NAME').alias('institution_name'),
    coalesce(col('SCHOOL_DSTRCT_CD'), col('SCHOOL_DISTRCT_CD')).alias('district_code'),
    col('SCHOOL_DSTRCT_NM').alias('district_name'),
    col('SUBGRP_DESC').alias('subgroup'),
    col('TEST_CMPNT_TYP_CD').alias('test_component'),
    
    # Composite join key for institution
    concat_ws('_', coalesce(col('SCHOOL_DSTRCT_CD'), col('SCHOOL_DISTRCT_CD')), col('INSTN_NUMBER')).alias('institution_key'),
    
    # Counts - strip trailing .0 and cast to int, handle TFS suppression
    when(col('NATIONAL_NUM_TESTED_CNT') == 'TFS', None)
        .when(col('NATIONAL_NUM_TESTED_CNT').isNull(), None)
        .otherwise(regexp_replace(col('NATIONAL_NUM_TESTED_CNT'), r'\.0*$', '')).cast(IntegerType()).alias('national_num_tested'),
    when(col('STATE_NUM_TESTED_CNT') == 'TFS', None)
        .when(col('STATE_NUM_TESTED_CNT').isNull(), None)
        .otherwise(regexp_replace(col('STATE_NUM_TESTED_CNT'), r'\.0*$', '')).cast(IntegerType()).alias('state_num_tested'),
    when(col('DSTRCT_NUM_TESTED_CNT') == 'TFS', None)
        .when(col('DSTRCT_NUM_TESTED_CNT').isNull(), None)
        .otherwise(regexp_replace(col('DSTRCT_NUM_TESTED_CNT'), r'\.0*$', '')).cast(IntegerType()).alias('district_num_tested'),
    when(col('INSTN_NUM_TESTED_CNT') == 'TFS', None)
        .when(col('INSTN_NUM_TESTED_CNT').isNull(), None)
        .otherwise(regexp_replace(col('INSTN_NUM_TESTED_CNT'), r'\.0*$', '')).cast(IntegerType()).alias('institution_num_tested'),
    
    # Scores - handle TFS suppression, cast to double
    when(col('NATIONAL_AVG_SCORE_VAL') == 'TFS', None)
        .when(col('NATIONAL_AVG_SCORE_VAL').isNull(), None)
        .otherwise(col('NATIONAL_AVG_SCORE_VAL')).cast(DoubleType()).alias('national_avg_score'),
    when(col('STATE_AVG_SCORE_VAL') == 'TFS', None)
        .when(col('STATE_AVG_SCORE_VAL').isNull(), None)
        .otherwise(col('STATE_AVG_SCORE_VAL')).cast(DoubleType()).alias('state_avg_score'),
    when(col('DSTRCT_AVG_SCORE_VAL') == 'TFS', None)
        .when(col('DSTRCT_AVG_SCORE_VAL').isNull(), None)
        .otherwise(col('DSTRCT_AVG_SCORE_VAL')).cast(DoubleType()).alias('district_avg_score'),
    when(col('INSTN_AVG_SCORE_VAL') == 'TFS', None)
        .when(col('INSTN_AVG_SCORE_VAL').isNull(), None)
        .otherwise(col('INSTN_AVG_SCORE_VAL')).cast(DoubleType()).alias('institution_avg_score'),
    
    col('#ASSMT_CD').alias('assessment_code'),
    col('HIGHEST_RECENT_IND').alias('highest_recent_indicator'),
    col('source_year'),
    col('source_file')
)

print(f"Silver table built: {act_silver.count():,} rows")
print("\nSample (Composite, 2024-25, All Students):")
act_silver.filter(
    "test_component = 'Composite' AND subgroup = 'All Students' AND school_year = '2024-25'"
).select(
    'institution_name', 'district_name', 'institution_avg_score', 'institution_num_tested'
).show(5, truncate=False)

In [0]:
from pyspark.sql.functions import min as spark_min, max as spark_max, avg as spark_avg, abs as spark_abs

print("=== Writing silver table ===\n")

act_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.act")

print("✓ workspace.silver.act written successfully")

print("\n=== Validation & Quality Checks ===\n")

# Check business key uniqueness
print("Checking business key uniqueness...")
key_check = act_silver.groupBy(
    'school_year', 'district_code', 'institution_number', 
    'subgroup', 'test_component'
).agg(cnt('*').alias('key_count')).filter('key_count > 1')

if key_check.count() > 0:
    print("⚠️  WARNING: Duplicate business keys found after deduplication!")
    key_check.show(5, truncate=False)
else:
    print("✓ All business keys are unique")

# TFS suppression analysis
print("\nTFS suppression analysis:")
total_records = act_silver.count()
tfs_num_tested = act_silver.filter("institution_num_tested IS NULL").count()
tfs_avg_score = act_silver.filter("institution_avg_score IS NULL").count()

print(f"  Total records: {total_records:,}")
print(f"  TFS suppressed (institution_num_tested): {tfs_num_tested:,} ({100*tfs_num_tested/total_records:.1f}%)")
print(f"  TFS suppressed (institution_avg_score): {tfs_avg_score:,} ({100*tfs_avg_score/total_records:.1f}%)")

# Score range validation (ACT valid range: 1-36)
print("\nScore range validation (ACT valid range: 1-36):")
score_ranges = act_silver.filter(
    "test_component = 'Composite' AND institution_avg_score IS NOT NULL"
).agg(
    spark_min('institution_avg_score').alias('min_score'),
    spark_max('institution_avg_score').alias('max_score')
).collect()[0]

print(f"  Min composite score: {score_ranges['min_score']}")
print(f"  Max composite score: {score_ranges['max_score']}")
if 1 <= score_ranges['min_score'] and score_ranges['max_score'] <= 36:
    print("✓ All scores within valid range")
else:
    print("⚠️  WARNING: Scores outside expected range (1-36)")
    act_silver.filter(
        "test_component = 'Composite' AND (institution_avg_score < 1 OR institution_avg_score > 36)"
    ).select('school_year', 'institution_name', 'test_component', 'institution_avg_score').show(5)

# Composite score consistency check
print("\nComposite score consistency check:")
print("Validating that composite ≈ average of (English + Math + Reading + Science)...")

# Pivot to get all component scores for each school/year/subgroup
from pyspark.sql import Window
import pyspark.sql.functions as F

composite_check = act_silver.filter(
    "institution_avg_score IS NOT NULL AND school_year >= '2020-21'"
).groupBy('school_year', 'institution_key', 'institution_name', 'subgroup').pivot('test_component').agg(
    F.first('institution_avg_score')
).filter(
    "Composite IS NOT NULL AND English IS NOT NULL AND Mathematics IS NOT NULL AND Reading IS NOT NULL AND Science IS NOT NULL"
).withColumn(
    'calculated_composite', (col('English') + col('Mathematics') + col('Reading') + col('Science')) / 4
).withColumn(
    'diff', spark_abs(col('Composite') - col('calculated_composite'))
).filter('diff > 0.5')

inconsistent_composite = composite_check.count()
if inconsistent_composite > 0:
    print(f"⚠️  WARNING: {inconsistent_composite:,} records where composite differs from component average by >0.5")
    composite_check.select(
        'school_year', 'institution_name', 'subgroup', 'Composite', 'calculated_composite', 'diff'
    ).orderBy(col('diff').desc()).show(5, truncate=False)
else:
    print("✓ All composite scores match component averages (within 0.5 rounding tolerance)")

# Test component distribution
print("\nTest component distribution:")
act_silver.groupBy('test_component').agg(cnt('*').alias('record_count')).orderBy('test_component').show(truncate=False)

# Year coverage validation
print("\nYear coverage validation:")
year_dist = act_silver.groupBy('school_year').agg(cnt('*').alias('record_count')).orderBy('school_year')
print("Expected: Relatively stable record counts per year")
year_dist.show(20, truncate=False)

# Subgroup coverage by year (check for reporting changes)
print("\nSubgroup coverage by year (recent years):")
act_silver.filter("school_year >= '2020-21'") \
    .groupBy('school_year', 'subgroup').agg(cnt('*').alias('records')) \
    .orderBy('school_year', col('records').desc()).show(50, truncate=False)

print("\n✓ Silver table ready for gold layer transformations")